# Cleanup: S3 to GCS Lineage Demo

This notebook cleans up all resources created by `setup/07_import_s3_to_gcs_with_lineage.ipynb`.

**Resources that will be deleted:**

1. **Data Lineage Resources:**
   - Lineage Events (S3→GCS and GCS→BigQuery)
   - Lineage Runs (2 runs)
   - Lineage Processes (2 processes)

2. **Dataplex Custom Catalog:**
   - S3 Custom Entry
   - Entry Type: `s3-object`
   - Entry Group: `aws-storage-entries`

3. **BigQuery:**
   - Table: `census_bureau_acs.s3_acs_data`

4. **Cloud Storage:**
   - Bucket: `{project-id}-census-s3-import` (and all contents)

**⚠️  WARNING:**
- This operation cannot be undone
- All data and metadata will be permanently deleted
- The BigQuery dataset `census_bureau_acs` will NOT be deleted (it may contain other tables)

**Required permissions:**
- `roles/storage.admin`
- `roles/bigquery.admin`
- `roles/dataplex.catalogAdmin`
- `roles/datacatalog.admin`

**Time Estimate:** 5-10 minutes

## Section 1: Configuration & Authentication

In [ ]:
# Configuration variables (must match setup notebook)
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
CATALOG_LOCATION = "us"

# Resource identifiers
BUCKET_NAME = f"{PROJECT_ID}-census-s3-import"
BQ_DATASET_ID = "census_bureau_acs"
BQ_TABLE_ID = "s3_acs_data"
ENTRY_GROUP_ID = "aws-storage-entries"
ENTRY_TYPE_ID = "s3-object"
LINEAGE_PROCESS_ID = "s3-to-gcs-import-process"
BQ_LINEAGE_PROCESS_ID = "gcs-to-bigquery-load-process"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print()
print("  Resources to delete:")
print(f"    - GCS Bucket: {BUCKET_NAME}")
print(f"    - BigQuery Table: {BQ_DATASET_ID}.{BQ_TABLE_ID}")
print(f"    - Entry Group: {ENTRY_GROUP_ID}")
print(f"    - Entry Type: {ENTRY_TYPE_ID}")
print(f"    - Lineage Processes: {LINEAGE_PROCESS_ID}, {BQ_LINEAGE_PROCESS_ID}")

In [ ]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

In [ ]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = [
    "google-cloud-storage",
    "google-cloud-bigquery",
    "google-cloud-dataplex",
    "google-cloud-datacatalog-lineage"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

In [ ]:
# Initialize clients
from google.cloud import storage, bigquery, dataplex_v1
from google.cloud.datacatalog_lineage_v1 import LineageClient
from google.api_core import exceptions

storage_client = storage.Client(project=PROJECT_ID)
bq_client = bigquery.Client(project=PROJECT_ID)
catalog_client = dataplex_v1.CatalogServiceClient()
lineage_client = LineageClient()

print("✅ Clients initialized:")

## Section 2: Delete Data Lineage Resources

Delete lineage processes (which cascades to delete runs and events).

In [ ]:
# Delete S3 to GCS Lineage Process
print("🗑️  Deleting S3 to GCS Lineage Process...")
print()

process_name = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/processes/{LINEAGE_PROCESS_ID}"

try:
    lineage_client.delete_process(name=process_name)
    print(f"✅ Process deleted: {process_name}")
    print("   (Associated runs and events are also deleted)")
except exceptions.NotFound:
    print(f"⚠️  Process not found (may have been already deleted): {process_name}")
except Exception as e:
    print(f"⚠️  Error deleting process: {e}")
    print("   Continuing with cleanup...")

In [ ]:
# Delete GCS to BigQuery Lineage Process
print("🗑️  Deleting GCS to BigQuery Lineage Process...")
print()

bq_process_name = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/processes/{BQ_LINEAGE_PROCESS_ID}"

try:
    lineage_client.delete_process(name=bq_process_name)
    print(f"✅ Process deleted: {bq_process_name}")
    print("   (Associated runs and events are also deleted)")
except exceptions.NotFound:
    print(f"⚠️  Process not found (may have been already deleted): {bq_process_name}")
except Exception as e:
    print(f"⚠️  Error deleting process: {e}")
    print("   Continuing with cleanup...")

## Section 3: Delete Dataplex Custom Catalog Resources

Delete S3 entries, entry type, and entry group.

In [ ]:
# Delete all entries in the entry group
print("🗑️  Deleting all S3 entries from entry group...")
print()

entry_group_path = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/entryGroups/{ENTRY_GROUP_ID}"

try:
    # List all entries in the group
    request = dataplex_v1.ListEntriesRequest(parent=entry_group_path)
    entries = catalog_client.list_entries(request=request)
    
    deleted_count = 0
    for entry in entries:
        try:
            catalog_client.delete_entry(name=entry.name)
            print(f"   ✅ Deleted entry: {entry.name.split('/')[-1]}")
            deleted_count += 1
        except Exception as e:
            print(f"   ⚠️  Error deleting entry {entry.name}: {e}")
    
    if deleted_count > 0:
        print()
        print(f"✅ Deleted {deleted_count} entry(ies)")
    else:
        print("⚠️  No entries found to delete")
        
except exceptions.NotFound:
    print(f"⚠️  Entry group not found: {ENTRY_GROUP_ID}")
except Exception as e:
    print(f"⚠️  Error listing/deleting entries: {e}")
    print("   Continuing with cleanup...")

In [ ]:
# Delete entry type
print("🗑️  Deleting entry type...")
print()

entry_type_path = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/entryTypes/{ENTRY_TYPE_ID}"

try:
    operation = catalog_client.delete_entry_type(name=entry_type_path)
    operation.result()  # Wait for operation to complete
    print(f"✅ Entry type deleted: {ENTRY_TYPE_ID}")
except exceptions.NotFound:
    print(f"⚠️  Entry type not found: {ENTRY_TYPE_ID}")
except Exception as e:
    print(f"⚠️  Error deleting entry type: {e}")
    print("   Continuing with cleanup...")

In [ ]:
# Delete entry group
print("🗑️  Deleting entry group...")
print()

try:
    operation = catalog_client.delete_entry_group(name=entry_group_path)
    operation.result()  # Wait for operation to complete
    print(f"✅ Entry group deleted: {ENTRY_GROUP_ID}")
except exceptions.NotFound:
    print(f"⚠️  Entry group not found: {ENTRY_GROUP_ID}")
except Exception as e:
    print(f"⚠️  Error deleting entry group: {e}")
    print("   Continuing with cleanup...")

## Section 4: Delete BigQuery Table

In [ ]:
# Delete BigQuery table
print("🗑️  Deleting BigQuery table...")
print()

table_id = f"{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}"

try:
    bq_client.delete_table(table_id, not_found_ok=True)
    print(f"✅ Table deleted: {table_id}")
except Exception as e:
    print(f"⚠️  Error deleting table: {e}")
    print("   Continuing with cleanup...")

## Section 5: Delete GCS Bucket

Delete the Cloud Storage bucket and all its contents.

In [ ]:
# Delete GCS bucket and contents
print("🗑️  Deleting GCS bucket and all contents...")
print()

try:
    bucket = storage_client.bucket(BUCKET_NAME)
    
    if bucket.exists():
        # Delete all objects in bucket
        blobs = list(bucket.list_blobs())
        if blobs:
            print(f"   Deleting {len(blobs)} object(s)...")
            for blob in blobs:
                blob.delete()
                print(f"   ✅ Deleted: {blob.name}")
        
        # Delete bucket
        bucket.delete()
        print()
        print(f"✅ Bucket deleted: {BUCKET_NAME}")
    else:
        print(f"⚠️  Bucket not found: {BUCKET_NAME}")
        
except Exception as e:
    print(f"⚠️  Error deleting bucket: {e}")
    print("   You may need to delete manually")

## Section 6: Cleanup Summary

In [ ]:
# Cleanup summary
print()
print("=" * 70)
print("🎉 CLEANUP COMPLETE")
print("=" * 70)
print()
print("✅ Successfully deleted:")
print()
print("   1️⃣  Data Lineage Resources:")
print("      - S3 to GCS Process (and associated runs/events)")
print("      - GCS to BigQuery Process (and associated runs/events)")
print()
print("   2️⃣  Dataplex Custom Catalog:")
print("      - S3 Custom Entries")
print(f"      - Entry Type: {ENTRY_TYPE_ID}")
print(f"      - Entry Group: {ENTRY_GROUP_ID}")
print()
print("   3️⃣  BigQuery:")
print(f"      - Table: {BQ_DATASET_ID}.{BQ_TABLE_ID}")
print()
print("   4️⃣  Cloud Storage:")
print(f"      - Bucket: {BUCKET_NAME} (and all contents)")
print()
print("=" * 70)
print()
print("ℹ️  Note:")
print(f"   - The BigQuery dataset '{BQ_DATASET_ID}' was NOT deleted")
print("   - Other resources from previous notebooks are NOT affected")
print()
print("✅ Cleanup complete! No ongoing charges from this demo.")